In [1]:
from sedona.spark import SedonaContext
import os
from time import time
from sedona.spark import dataframe_to_arrow
from sedona.spark.geoarrow.geoarrow import create_spatial_dataframe

## GeoPandas DataFrame from Sedona Spatial DataFrame

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/07 18:16:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/07 18:17:01 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/07 18:17:01 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/07 18:17:01 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/07 18:17:01 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/12/07 18:17:01 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/07 18:17:01 WARN SimpleFunctionRegistry: The function st_envelop

In [3]:
sedona.read.format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/transportation_barcelona/barcelona.geoparquet").columns

['id',
 'geometry',
 'bbox',
 'version',
 'sources',
 'subtype',
 'class',
 'names',
 'connectors',
 'routes',
 'subclass',
 'subclass_rules',
 'access_restrictions',
 'level_rules',
 'destinations',
 'prohibited_transitions',
 'road_surface',
 'road_flags',
 'speed_limits',
 'width_rules']

In [4]:
df = sedona.read.format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/transportation_barcelona/barcelona.geoparquet").\
    select("id", "bbox", "version", "sources", "geometry", "subtype", "class", "names", "connectors", "routes", "subclass", "level_rules", "road_surface", "road_flags", "width_rules")

In [5]:
df.columns

['id',
 'bbox',
 'version',
 'sources',
 'geometry',
 'subtype',
 'class',
 'names',
 'connectors',
 'routes',
 'subclass',
 'level_rules',
 'road_surface',
 'road_flags',
 'width_rules']

# Sedona DataFrame to GeoPandas

In [6]:
## Inefficient way of converting spatial DataFrame to GeoPandas

In [7]:
import geopandas as gpd

start = time()
gdf = gpd.GeoDataFrame(df.toPandas(), geometry="geometry")
print(f"converted in {time() - start}")

converted in 24.009087800979614


In [8]:
## using the GeoArrow conversion

In [9]:
start = time()
gdf = gpd.GeoDataFrame.from_arrow(dataframe_to_arrow(df))
print(f"converted in {time() - start}")

converted in 17.521244287490845


# Creating Sedona DataFrame from shapely objects

In [10]:
from shapely.geometry import Point
import sedona.spark.sql.types as st
import pyspark.sql.types as t
 
schema = t.StructType(
    [
        t.StructField("id", t.IntegerType()),
        t.StructField("geom", st.GeometryType()),
    ]
)
 
shapely_df = sedona.createDataFrame([
    [1, Point(21, 52)],
    [2, Point(21, 45)]
], schema=schema)

In [11]:
shapely_df.show()

+---+-------------+
| id|         geom|
+---+-------------+
|  1|POINT (21 52)|
|  2|POINT (21 45)|
+---+-------------+



In [12]:
sedona.createDataFrame([
    {"id": 1, "geom": Point(21, 52)},
    {"id": 2, "geom": Point(21, 45)}
]).show()

+-------------+---+
|         geom| id|
+-------------+---+
|POINT (21 52)|  1|
|POINT (21 45)|  2|
+-------------+---+



# Sedona DataFrame from GeoPandas

In [13]:
gdf_subset = gdf[["id", "bbox", "version", "subtype", "class", "geometry", "names"]]

In [14]:
start = time()
sedona.createDataFrame(gdf_subset)
print(f"converted in {time() - start}")

converted in 2.2353267669677734


In [15]:
# Sedona DataFrame from geopandas using Apache Arrow

In [16]:
start = time()
create_spatial_dataframe(sedona, gdf_subset)
print(f"converted in {time() - start}")

converted in 0.9804487228393555


# Writing own UDF function

In [17]:
import pyspark.sql.functions as f
import sedona.spark.sql.types as st
import shapely.geometry.base as b
 
def create_buffer_distance(
    s: b.BaseGeometry,
    distance_from: float,
    distance_to: float
) -> b.BaseGeometry:
    buffer_a = s.buffer(distance_from)
    buffer_b = s.buffer(distance_to)
    return buffer_b.difference(buffer_a)
 
buffer_distanced_udf = f.udf(create_buffer_distance, st.GeometryType())
 
sedona.udf.register(
    "ST_BufferDistanceNonVectorized",
    buffer_distanced_udf
)

In [18]:
df.createOrReplaceTempView("roads")

In [19]:
sedona.sql(
"""
    SELECT 
        ST_BufferDistanceNonVectorized(
            geometry,
            CAST(0.0001 AS FLOAT),
            CAST(0.0002 AS FLOAT)
        ) AS geometry
    FROM roads
    """
).show()

[Stage 13:>                                                         (0 + 1) / 1]

+--------------------+
|            geometry|
+--------------------+
|POLYGON ((3.70448...|
|POLYGON ((5.33776...|
|POLYGON ((2.18222...|
|POLYGON ((3.70527...|
|POLYGON ((3.06424...|
|POLYGON ((8.45250...|
|POLYGON ((-0.6447...|
|POLYGON ((8.91493...|
|POLYGON ((1.44936...|
|POLYGON ((2.63515...|
|POLYGON ((2.08226...|
|POLYGON ((2.08236...|
|POLYGON ((2.08817...|
|POLYGON ((2.08926...|
|POLYGON ((2.08997...|
|POLYGON ((2.09033...|
|POLYGON ((2.09064...|
|POLYGON ((2.09172...|
|POLYGON ((2.09142...|
|POLYGON ((2.09272...|
+--------------------+
only showing top 20 rows



# Writing Vectorized UDF (better performance)

In [20]:
from sedona.spark.sql.functions import sedona_vectorized_udf
from sedona.spark.sql.types import GeometryType

@sedona_vectorized_udf(return_type=GeometryType())
def vectorized_symmetrical_buffer_distance_udf(
        geom: b.BaseGeometry
) -> b.BaseGeometry:
    return create_buffer_distance(geom, 0.0001, 0.0002)

In [21]:
df.select(
    vectorized_symmetrical_buffer_distance_udf(f.col("geometry"))
).show(10)

[Stage 14:>                                                         (0 + 1) / 1]

+------------------------------+
|SedonaPandasArrowUDF(geometry)|
+------------------------------+
|          POLYGON ((3.70448...|
|          POLYGON ((5.33776...|
|          POLYGON ((2.18222...|
|          POLYGON ((3.70527...|
|          POLYGON ((3.06424...|
|          POLYGON ((8.45250...|
|          POLYGON ((-0.6447...|
|          POLYGON ((8.91493...|
|          POLYGON ((1.44936...|
|          POLYGON ((2.63515...|
+------------------------------+
only showing top 10 rows

